In [3]:
"""
Quantile Regression VaR forecasting

- Reads SPY dataset (log_ret, rv5, bv)
- Builds one-day-ahead target y_t = r_{t+1}
- Uses features available at time t: [r_t, rv5_t, bv_t] (+ intercept)
- Fits quantile regression in an expanding or rolling window
- Generates one-step-ahead VaR forecasts for alpha in {0.01, 0.05, 0.10}

Output:
    forecasts_df: a DataFrame containing actual next-day returns and VaR forecasts
"""

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import chi2

In [4]:

# ----------------------------
# Step 1: Load and clean data
# ----------------------------
DATA_PATH = "./spy_data.csv"
df = pd.read_csv(DATA_PATH)

# Keep only required columns
required_cols = ["log_ret", "rv5", "bv"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns in CSV: {missing}")

df = df[required_cols].dropna().copy()


# ------------------------------------------
# Step 0: Define one-day-ahead target (y_t)
# ------------------------------------------
# y_t corresponds to next-day return r_{t+1}
df["y_next_ret"] = df["log_ret"].shift(-1)

# ----------------------------
# Step 1: Define features X_t
# ----------------------------
# Features available at time t:
# - r_t (today's return)
# - rv5_t (today's realized volatility proxy)
# - bv_t (today's bipower variation)
df["x_ret"] = df["log_ret"]
df["x_rv5"] = df["rv5"]
df["x_bv"] = df["bv"]

# Optionally stabilize scale (often useful);、
# df["x_rv5"] = np.log(df["rv5"].clip(lower=1e-12))
# df["x_bv"]  = np.log(df["bv"].clip(lower=1e-12))

# Drop last row (because y_next_ret is NaN there after shift)
df_model = df.dropna(subset=["y_next_ret", "x_ret", "x_rv5", "x_bv"]).copy()


In [9]:
print(df_model.head())

    log_ret       rv5        bv  y_next_ret     x_ret     x_rv5      x_bv
0 -0.038711  0.000224  0.000173    0.002192 -0.038711  0.000224  0.000173
1  0.002192  0.000314  0.000303    0.000692  0.002192  0.000314  0.000303
2  0.000692  0.000131  0.000128    0.026571  0.000692  0.000131  0.000128
3  0.026571  0.000094  0.000079    0.011043  0.026571  0.000094  0.000079
4  0.011043  0.000120  0.000136   -0.012787  0.011043  0.000120  0.000136


In [5]:


# ------------------------------------------------------------
# Steps 2–6: Quantile regression + one-step-ahead VaR forecasts
# ------------------------------------------------------------
def quantile_var_forecast(
    data: pd.DataFrame,
    alphas=(0.01, 0.05, 0.10),
    train_start: int = 0,
    initial_train_size: int = 1000,
    window_type: str = "expanding",  # "expanding" or "rolling"
    rolling_window_size: int = 1500,
) -> pd.DataFrame:
    """
    Fits quantile regression models in a walk-forward manner and produces VaR forecasts.

    Parameters
    ----------
    data : DataFrame
        Must contain columns: y_next_ret, x_ret, x_rv5, x_bv.
    alphas : tuple
        Quantiles to forecast.
    initial_train_size : int
        First forecast starts after this many observations for training.
    window_type : str
        "expanding" uses all data from train_start to t,
        "rolling" uses a fixed-size window of rolling_window_size ending at t.
    rolling_window_size : int
        Window size if window_type == "rolling".

    Returns
    -------
    out : DataFrame
        Index aligned with forecast dates (time t, forecasting t+1 return).
        Columns include:
            - y_next_ret (realized next-day return)
            - VaR_{alpha} (forecasted conditional quantile)
    """
    if window_type not in ("expanding", "rolling"):
        raise ValueError("window_type must be 'expanding' or 'rolling'")

    # Prepare arrays
    y = data["y_next_ret"]
    X = data[["x_ret", "x_rv5", "x_bv"]]
    X = sm.add_constant(X, has_constant="add")  # add intercept

    n = len(data)
    start_forecast = max(train_start + initial_train_size, 1)

    # Output container
    out = pd.DataFrame(index=data.index, data={"y_next_ret": y.values})
    for a in alphas:
        out[f"VaR_{a:.2f}"] = np.nan


## Walk-forward quantile regression VaR forecasts

In [10]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

def quantile_var_forecast(
    data: pd.DataFrame,
    alphas=(0.01, 0.05, 0.10),
    train_start: int = 0,
    initial_train_size: int = 1000,
    window_type: str = "rolling",   # "expanding" or "rolling"
    rolling_window_size: int = 1500,
) -> pd.DataFrame:
    """
    Walk-forward quantile regression VaR forecasts.
    Returns a DataFrame with y_next_ret and VaR_{alpha} columns.
    """

    if window_type not in ("expanding", "rolling"):
        raise ValueError("window_type must be 'expanding' or 'rolling'")

    required = ["y_next_ret", "x_ret", "x_rv5", "x_bv"]
    missing = [c for c in required if c not in data.columns]
    if missing:
        raise ValueError(f"Missing columns in input data: {missing}")

    y = data["y_next_ret"].astype(float)
    X = data[["x_ret", "x_rv5", "x_bv"]].astype(float)
    X = sm.add_constant(X, has_constant="add")

    n = len(data)

    # For rolling windows, you need enough history to fill the window.
    start_forecast = train_start + initial_train_size
    if window_type == "rolling":
        start_forecast = max(start_forecast, train_start + rolling_window_size)

    if start_forecast >= n:
        raise ValueError(
            f"Not enough data to start forecasting. n={n}, start_forecast={start_forecast}. "
            f"Reduce initial_train_size or rolling_window_size."
        )

    # Output container
    out = pd.DataFrame(index=data.index, data={"y_next_ret": y.values})
    for a in alphas:
        out[f"VaR_{a:.2f}"] = np.nan

    # Walk-forward forecasting
    for i in range(start_forecast, n):
        if window_type == "expanding":
            train_slice = slice(train_start, i)
        else:
            train_left = max(train_start, i - rolling_window_size)
            train_slice = slice(train_left, i)

        X_train = X.iloc[train_slice]
        y_train = y.iloc[train_slice]
        X_pred = X.iloc[i:i+1]

        for a in alphas:
            model = sm.QuantReg(y_train, X_train)
            res = model.fit(q=a, max_iter=10000)
            out.loc[out.index[i], f"VaR_{a:.2f}"] = float(res.predict(X_pred).iloc[0])

    # Keep only rows with all VaR forecasts
    out = out.dropna(subset=[f"VaR_{a:.2f}" for a in alphas])

    if out.empty:
        raise ValueError("No VaR forecasts were generated (out is empty).")

    return out


In [11]:
forecasts_df = quantile_var_forecast(
    data=df_model,
    alphas=(0.01, 0.05, 0.10),
    train_start=0,
    initial_train_size=1000,
    window_type="rolling",
    rolling_window_size=1500,)

D:\myanaconda\envs\TORCHGPU3\lib\site-packages\statsmodels\regression\quantile_regression.py:191: IterationLimitWarning: Maximum number of iterations (10000) reached.
  warnings.warn("Maximum number of iterations (" + str(max_iter) +
D:\myanaconda\envs\TORCHGPU3\lib\site-packages\statsmodels\regression\quantile_regression.py:191: IterationLimitWarning: Maximum number of iterations (10000) reached.
  warnings.warn("Maximum number of iterations (" + str(max_iter) +
D:\myanaconda\envs\TORCHGPU3\lib\site-packages\statsmodels\regression\quantile_regression.py:191: IterationLimitWarning: Maximum number of iterations (10000) reached.
  warnings.warn("Maximum number of iterations (" + str(max_iter) +
D:\myanaconda\envs\TORCHGPU3\lib\site-packages\statsmodels\regression\quantile_regression.py:191: IterationLimitWarning: Maximum number of iterations (10000) reached.
  warnings.warn("Maximum number of iterations (" + str(max_iter) +
D:\myanaconda\envs\TORCHGPU3\lib\site-packages\statsmodels\regre

In [12]:
print(type(forecasts_df))
print(forecasts_df is None)


<class 'pandas.core.frame.DataFrame'>
False


In [ ]:
"""
Step 7: VaR Backtesting and Evaluation

Implements:
- Violation indicators
- Empirical coverage rates
- Kupiec unconditional coverage test
- Christoffersen conditional coverage test
- Average quantile (pinball) loss

Input:
    forecasts_df produced in Steps 0–6
"""

import numpy as np
import pandas as pd



# -----------------------------------
# Step 7.1: Violation indicator
# -----------------------------------
def compute_violations(forecasts: pd.DataFrame, alpha: float):
    """
    Compute VaR violations for a given quantile level.

    A violation occurs if realized return < forecasted VaR.
    """
    var_col = f"VaR_{alpha:.2f}"
    violations = (forecasts["y_next_ret"] < forecasts[var_col]).astype(int)
    return violations


# -----------------------------------
# Step 7.2: Kupiec Unconditional Coverage Test
# -----------------------------------
def kupiec_test(violations: np.ndarray, alpha: float):
    """
    Kupiec (1995) unconditional coverage likelihood-ratio test.

    H0: P(violation) = alpha
    """
    T = len(violations)
    x = violations.sum()

    # Empirical violation probability
    pi_hat = x / T

    # Log-likelihoods
    ll_null = (T - x) * np.log(1 - alpha) + x * np.log(alpha)
    ll_alt = (T - x) * np.log(1 - pi_hat) + x * np.log(pi_hat)

    LR_uc = -2 * (ll_null - ll_alt)
    p_value = 1 - chi2.cdf(LR_uc, df=1)

    return {
        "LR_uc": LR_uc,
        "p_value": p_value,
        "empirical_rate": pi_hat
    }


# -----------------------------------
# Step 7.3: Christoffersen Conditional Coverage Test
# -----------------------------------
def christoffersen_test(violations: np.ndarray, alpha: float):
    """
    Christoffersen (1998) conditional coverage test.

    Tests:
    - Correct unconditional coverage
    - Independence of violations
    """
    v = violations.astype(int)

    # Transition counts
    n00 = np.sum((v[:-1] == 0) & (v[1:] == 0))
    n01 = np.sum((v[:-1] == 0) & (v[1:] == 1))
    n10 = np.sum((v[:-1] == 1) & (v[1:] == 0))
    n11 = np.sum((v[:-1] == 1) & (v[1:] == 1))

    # Transition probabilities
    pi0 = n01 / (n00 + n01) if (n00 + n01) > 0 else 0.0
    pi1 = n11 / (n10 + n11) if (n10 + n11) > 0 else 0.0
    pi = (n01 + n11) / (n00 + n01 + n10 + n11)

    # Log-likelihoods
    ll_ind = (
        n00 * np.log(1 - pi0) + n01 * np.log(pi0)
        + n10 * np.log(1 - pi1) + n11 * np.log(pi1)
    )

    ll_null = (
        (n00 + n10) * np.log(1 - pi)
        + (n01 + n11) * np.log(pi)
    )

    LR_cc = -2 * (ll_null - ll_ind)
    p_value = 1 - chi2.cdf(LR_cc, df=1)

    return {
        "LR_cc": LR_cc,
        "p_value": p_value
    }


# -----------------------------------
# Step 7.4: Quantile (Pinball) Loss
# -----------------------------------
def quantile_loss(y: np.ndarray, q_hat: np.ndarray, alpha: float):
    """
    Average pinball loss for quantile alpha.
    """
    u = y - q_hat
    loss = np.maximum(alpha * u, (alpha - 1) * u)
    return np.mean(loss)


In [13]:


# -----------------------------------
# Run backtests for each VaR level
# -----------------------------------
results = {}

for alpha in (0.01, 0.05, 0.10):
    violations = compute_violations(forecasts_df, alpha)

    kupiec_res = kupiec_test(violations.values, alpha)
    christoffersen_res = christoffersen_test(violations.values, alpha)

    qloss = quantile_loss(
        forecasts_df["y_next_ret"].values,
        forecasts_df[f"VaR_{alpha:.2f}"].values,
        alpha
    )

    results[alpha] = {
        "violation_rate": kupiec_res["empirical_rate"],
        "kupiec_LR": kupiec_res["LR_uc"],
        "kupiec_p": kupiec_res["p_value"],
        "christoffersen_LR": christoffersen_res["LR_cc"],
        "christoffersen_p": christoffersen_res["p_value"],
        "quantile_loss": qloss
    }


# -----------------------------------
# Present results in a clean table
# -----------------------------------
results_df = pd.DataFrame(results).T
print(results_df)


      violation_rate  kupiec_LR  kupiec_p  christoffersen_LR  \
0.01        0.014336   5.255274  0.021880           1.899491   
0.05        0.050335   0.007379  0.931547           0.132089   
0.10        0.094298   1.153891  0.282736           0.359974   

      christoffersen_p  quantile_loss  
0.01          0.168135       0.000396  
0.05          0.716276       0.001287  
0.10          0.548521       0.002055  


## GARCH-based VaR forecasting

In [22]:

# Requires: pip install arch
import numpy as np
import pandas as pd
from arch import arch_model
from scipy.stats import norm, t as student_t


def garch_var_forecast(
    data: pd.DataFrame,
    alphas=(0.01, 0.05, 0.10),
    train_start: int = 0,
    initial_train_size: int = 1000,
    window_type: str = "rolling",   # "expanding" or "rolling"
    rolling_window_size: int = 1500,
    dist: str = "t",                # "t" or "normal"
    mean: str = "Constant",         # "Zero" or "Constant"
    vol: str = "GARCH",             # keep as "GARCH"
    p: int = 1,
    q: int = 1,
    rescale_to_pct: bool = True,    # arch fits better with % returns
) -> pd.DataFrame:
    """
    Walk-forward 1-step-ahead VaR using AR(0)-GARCH(p,q) with Normal or Student-t innovations.

    Input data must contain:
      - y_next_ret : r_{t+1} (realized next-day return)
      - x_ret      : r_t (today's return)

    Output aligned to time t (forecasting t+1):
      - y_next_ret
      - VaR_{alpha} for each alpha
    """
    if window_type not in ("expanding", "rolling"):
        raise ValueError("window_type must be 'expanding' or 'rolling'")

    required = ["y_next_ret", "x_ret"]
    missing = [c for c in required if c not in data.columns]
    if missing:
        raise ValueError(f"Missing columns in input data: {missing}")

    r = data["x_ret"].astype(float).copy()
    y_next = data["y_next_ret"].astype(float).copy()

    # arch_model often expects returns in percent scale
    scale = 100.0 if rescale_to_pct else 1.0
    r_fit = r * scale

    n = len(data)
    start_forecast = train_start + initial_train_size
    if window_type == "rolling":
        start_forecast = max(start_forecast, train_start + rolling_window_size)
    if start_forecast >= n:
        raise ValueError(
            f"Not enough data to start forecasting. n={n}, start_forecast={start_forecast}. "
            f"Reduce initial_train_size or rolling_window_size."
        )

    out = pd.DataFrame(index=data.index, data={"y_next_ret": y_next.values})
    for a in alphas:
        out[f"VaR_{a:.2f}"] = np.nan

    for i in range(start_forecast, n):
        if window_type == "expanding":
            train_slice = slice(train_start, i)
        else:
            train_left = max(train_start, i - rolling_window_size)
            train_slice = slice(train_left, i)

        r_train = r_fit.iloc[train_slice]

        am = arch_model(
            r_train,
            mean=mean,
            vol=vol,
            p=p,
            q=q,
            dist=dist,
            rescale=False,  # we already rescaled explicitly
        )
        res = am.fit(disp="off")

        # 1-step ahead forecast (t -> t+1)
        f = res.forecast(horizon=1, reindex=False)
        mu_next = float(f.mean.iloc[-1, 0])
        var_next = float(f.variance.iloc[-1, 0])
        sigma_next = np.sqrt(max(var_next, 0.0))

        # Quantile of standardized innovation
        if dist.lower() == "t":
            nu = float(res.params.get("nu"))
            q_std = student_t.ppf(alphas, df=nu)
        elif dist.lower() in ("normal", "gaussian"):
            q_std = norm.ppf(alphas)
        else:
            raise ValueError("dist must be 't' or 'normal'")

        # VaR in the fitted scale (percent if rescale_to_pct=True)
        var_next_alpha = mu_next + sigma_next * np.asarray(q_std)

        # convert back to original return scale if needed
        var_next_alpha = var_next_alpha / scale

        for a, v in zip(alphas, var_next_alpha):
            out.loc[out.index[i], f"VaR_{a:.2f}"] = float(v)

    out = out.dropna(subset=[f"VaR_{a:.2f}" for a in alphas])
    if out.empty:
        raise ValueError("No VaR forecasts were generated (out is empty).")

    return out






In [23]:
# Example usage (assuming df_model already exists with y_next_ret and x_ret):
forecasts_df = garch_var_forecast(
    data=df_model,
    alphas=(0.01, 0.05, 0.10),
    initial_train_size=1000,
    window_type="rolling",
    rolling_window_size=1500,
    dist="t",
    mean="Constant",
    p=1, q=1,
    rescale_to_pct=True
)

In [16]:
# evaluate the results using the backtesting functions defined earlier
results = {}
for alpha in (0.01, 0.05, 0.10):
    violations = compute_violations(forecasts_df, alpha)

    kupiec_res = kupiec_test(violations.values, alpha)
    christoffersen_res = christoffersen_test(violations.values, alpha)

    qloss = quantile_loss(
        forecasts_df["y_next_ret"].values,
        forecasts_df[f"VaR_{alpha:.2f}"].values,
        alpha
    )

    results[alpha] = {
        "violation_rate": kupiec_res["empirical_rate"],
        "kupiec_LR": kupiec_res["LR_uc"],
        "kupiec_p": kupiec_res["p_value"],
        "christoffersen_LR": christoffersen_res["LR_cc"],
        "christoffersen_p": christoffersen_res["p_value"],
        "quantile_loss": qloss
    }
# -----------------------------------
# Present results in a clean table
# -----------------------------------
results_df = pd.DataFrame(results).T
print(results_df)

      violation_rate  kupiec_LR  kupiec_p  christoffersen_LR  \
0.01        0.009876   0.004915  0.944111           4.309946   
0.05        0.044600   1.996401  0.157673           0.499273   
0.10        0.080280  14.445206  0.000144           0.033653   

      christoffersen_p  quantile_loss  
0.01          0.037890       0.000361  
0.05          0.479820       0.001250  
0.10          0.854447       0.002003  


## Neural quantile regression

In [17]:

# Neural Quantile Regression (Plan B: warm-start) for VaR forecasting
# - Jointly forecasts multiple quantiles with non-crossing constraint (hard monotonicity)
# - Rolling/expanding walk-forward forecasting
# - Per-window feature standardization (fit on train window only; no leakage)
#
# Requires: pip install torch
import math
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F


# ----------------------------
# Utilities
# ----------------------------
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Determinism (may impact speed; safe default for reproducibility)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


class ZScoreScaler:
    """Simple per-window z-score scaler (fit on train only)."""
    def __init__(self, eps: float = 1e-12):
        self.eps = eps
        self.mean_ = None
        self.std_ = None

    def fit(self, X: np.ndarray) -> "ZScoreScaler":
        self.mean_ = X.mean(axis=0, keepdims=True)
        self.std_ = X.std(axis=0, keepdims=True)
        self.std_ = np.maximum(self.std_, self.eps)
        return self

    def transform(self, X: np.ndarray) -> np.ndarray:
        if self.mean_ is None or self.std_ is None:
            raise RuntimeError("Scaler has not been fit.")
        return (X - self.mean_) / self.std_


def pinball_loss(y: torch.Tensor, q: torch.Tensor, alpha: float) -> torch.Tensor:
    """
    y: (B, 1) or (B,)
    q: (B, 1) or (B,)
    """
    u = y - q
    return torch.maximum(alpha * u, (alpha - 1.0) * u).mean()


# ----------------------------
# Monotone multi-quantile MLP
# ----------------------------
class MonotoneQuantileMLP(nn.Module):
    """
    Outputs K quantiles with hard non-crossing constraints by construction:
      q1 = z1
      q2 = z1 + softplus(z2)
      q3 = z1 + softplus(z2) + softplus(z3)
      ...
    """
    def __init__(self, in_dim: int, hidden_dims=(32, 32), dropout: float = 0.0, K: int = 3):
        super().__init__()
        if K < 1:
            raise ValueError("K must be >= 1")

        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        self.backbone = nn.Sequential(*layers)

        # Raw outputs z in R^K (z1 baseline, z2..zK increments before softplus)
        self.head = nn.Linear(prev, K)
        self.K = K

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.backbone(x)
        z = self.head(h)  # (B, K)
        if self.K == 1:
            return z

        base = z[:, :1]  # (B, 1)
        inc_raw = z[:, 1:]  # (B, K-1)
        inc_pos = F.softplus(inc_raw)  # strictly positive
        cum_inc = torch.cumsum(inc_pos, dim=1)  # (B, K-1)
        qs = torch.cat([base, base + cum_inc], dim=1)  # (B, K)
        return qs


# ----------------------------
# Plan B Forecasting Function
# ----------------------------
def neural_quantile_var_forecast_planB(
    data: pd.DataFrame,
    alphas=(0.01, 0.05, 0.10),
    train_start: int = 0,
    initial_train_size: int = 1000,
    window_type: str = "rolling",         # "expanding" or "rolling"
    rolling_window_size: int = 1500,
    # model/optimization hyperparams
    hidden_dims=(32, 32),
    dropout: float = 0.0,
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    batch_size: int = 256,
    init_epochs: int = 80,                # full training at first step
    update_epochs: int = 5,               # warm-start updates for subsequent steps
    early_stop_patience: int = 10,         # applied to init training; updates use fixed epochs by default
    reinit_every: int | None = 200,        # periodic full re-init; None disables
    device: str | None = None,            # "cpu" or "cuda"; None = auto
    seed: int = 42,
    verbose_every: int | None = None,      # e.g., 200 to print progress; None = silent
) -> pd.DataFrame:
    """
    Walk-forward VaR forecasting with neural quantile regression (Plan B warm-start).

    Required columns in `data`:
      - y_next_ret : r_{t+1}
      - x_ret      : r_t
      - x_rv5      : rv5_t
      - x_bv       : bv_t

    Output:
      DataFrame with y_next_ret and VaR_{alpha} columns aligned to forecast times (t).
    """
    if window_type not in ("expanding", "rolling"):
        raise ValueError("window_type must be 'expanding' or 'rolling'")

    required = ["y_next_ret", "x_ret", "x_rv5", "x_bv"]
    missing = [c for c in required if c not in data.columns]
    if missing:
        raise ValueError(f"Missing columns in input data: {missing}")

    set_seed(seed)

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    # Extract arrays
    y_all = data["y_next_ret"].astype(float).to_numpy()
    X_all = data[["x_ret", "x_rv5", "x_bv"]].astype(float).to_numpy()

    n = len(data)
    start_forecast = train_start + initial_train_size
    if window_type == "rolling":
        start_forecast = max(start_forecast, train_start + rolling_window_size)
    if start_forecast >= n:
        raise ValueError(
            f"Not enough data to start forecasting. n={n}, start_forecast={start_forecast}. "
            f"Reduce initial_train_size or rolling_window_size."
        )

    # Output container
    out = pd.DataFrame(index=data.index, data={"y_next_ret": y_all})
    for a in alphas:
        out[f"VaR_{a:.2f}"] = np.nan

    # Map alphas to fixed output order (sorted)
    alphas = tuple(sorted(alphas))
    K = len(alphas)

    # Model state (warm-start)
    model: MonotoneQuantileMLP | None = None
    optimizer: torch.optim.Optimizer | None = None

    def make_model_and_optim() -> tuple[MonotoneQuantileMLP, torch.optim.Optimizer]:
        m = MonotoneQuantileMLP(
            in_dim=X_all.shape[1],
            hidden_dims=hidden_dims,
            dropout=dropout,
            K=K,
        ).to(device)
        opt = torch.optim.Adam(m.parameters(), lr=lr, weight_decay=weight_decay)
        return m, opt

    def train_one_model(
        m: MonotoneQuantileMLP,
        opt: torch.optim.Optimizer,
        Xtr: np.ndarray,
        ytr: np.ndarray,
        epochs: int,
        use_early_stop: bool,
    ) -> None:
        m.train()
        Xtr_t = torch.tensor(Xtr, dtype=torch.float32, device=device)
        ytr_t = torch.tensor(ytr, dtype=torch.float32, device=device).view(-1, 1)

        N = Xtr_t.shape[0]
        idx = torch.arange(N, device=device)

        best_loss = float("inf")
        best_state = None
        patience_left = early_stop_patience

        for ep in range(epochs):
            # shuffle
            perm = idx[torch.randperm(N)]
            total_loss = 0.0
            seen = 0

            for s in range(0, N, batch_size):
                b = perm[s : s + batch_size]
                xb = Xtr_t[b]
                yb = ytr_t[b]

                opt.zero_grad(set_to_none=True)
                qhat = m(xb)  # (B, K)

                # summed pinball across quantiles
                loss = 0.0
                for j, a in enumerate(alphas):
                    loss = loss + pinball_loss(yb, qhat[:, j:j+1], float(a))

                loss.backward()
                opt.step()

                bs = xb.shape[0]
                total_loss += float(loss.detach().cpu()) * bs
                seen += bs

            avg_loss = total_loss / max(seen, 1)

            if use_early_stop:
                # early stopping on training loss (simple, no validation split to keep pipeline minimal)
                if avg_loss + 1e-10 < best_loss:
                    best_loss = avg_loss
                    best_state = {k: v.detach().cpu().clone() for k, v in m.state_dict().items()}
                    patience_left = early_stop_patience
                else:
                    patience_left -= 1
                    if patience_left <= 0:
                        break

        if use_early_stop and best_state is not None:
            m.load_state_dict(best_state)

    # Walk-forward
    for i in range(start_forecast, n):
        if window_type == "expanding":
            train_slice = slice(train_start, i)
        else:
            train_left = max(train_start, i - rolling_window_size)
            train_slice = slice(train_left, i)

        X_train_raw = X_all[train_slice]
        y_train = y_all[train_slice]

        # Per-window scaling (fit on train only)
        scaler = ZScoreScaler().fit(X_train_raw)
        X_train = scaler.transform(X_train_raw)
        X_pred = scaler.transform(X_all[i:i+1])

        # Decide whether to reinitialize (periodic refresh) or warm-start update
        do_reinit = (model is None)
        if (not do_reinit) and (reinit_every is not None):
            step_count = i - start_forecast
            if step_count % int(reinit_every) == 0:
                do_reinit = True

        if do_reinit:
            model, optimizer = make_model_and_optim()
            # full(ish) training at (re)init
            train_one_model(
                model, optimizer,
                X_train, y_train,
                epochs=init_epochs,
                use_early_stop=True,
            )
        else:
            # warm-start update: few epochs, fixed (can be made adaptive if needed)
            train_one_model(
                model, optimizer,
                X_train, y_train,
                epochs=update_epochs,
                use_early_stop=False,
            )

        # Predict
        model.eval()
        with torch.no_grad():
            x_pred_t = torch.tensor(X_pred, dtype=torch.float32, device=device)
            q_pred = model(x_pred_t).detach().cpu().numpy().reshape(-1)  # (K,)

        # Store forecasts
        for j, a in enumerate(alphas):
            out.loc[out.index[i], f"VaR_{a:.2f}"] = float(q_pred[j])

        if verbose_every is not None and ((i - start_forecast) % int(verbose_every) == 0):
            print(f"[NQR-PlanB] step {i}/{n-1} stored forecasts.")

    # Keep only rows with forecasts
    out = out.dropna(subset=[f"VaR_{a:.2f}" for a in alphas])
    if out.empty:
        raise ValueError("No VaR forecasts were generated (out is empty).")

    return out



In [18]:

# ----------------------------
# Example usage (drop-in)
# ----------------------------
forecasts_df_nqr = neural_quantile_var_forecast_planB(
    data=df_model,
    alphas=(0.01, 0.05, 0.10),
    train_start=0,
    initial_train_size=1000,
    window_type="rolling",
    rolling_window_size=1500,
    hidden_dims=(32, 32),
    lr=1e-3,
    weight_decay=1e-4,
    init_epochs=80,
    update_epochs=5,
    reinit_every=200,    # set None to disable periodic re-init
    device=None,         # auto
    seed=42,
    verbose_every=200,
)
#
# Then reuse your existing backtest code:
# violations = compute_violations(forecasts_df_nqr, 0.01), etc.
results_nqr = {}
for alpha in (0.01, 0.05, 0.10):
    violations = compute_violations(forecasts_df_nqr, alpha)

    kupiec_res = kupiec_test(violations.values, alpha)
    christoffersen_res = christoffersen_test(violations.values, alpha)
    qloss = quantile_loss(
        forecasts_df_nqr["y_next_ret"].values,
        forecasts_df_nqr[f"VaR_{alpha:.2f}"].values,
        alpha
    )

    results_nqr[alpha] = {
        "violation_rate": kupiec_res["empirical_rate"],
        "kupiec_LR": kupiec_res["LR_uc"],
        "kupiec_p": kupiec_res["p_value"],
        "christoffersen_LR": christoffersen_res["LR_cc"],
        "christoffersen_p": christoffersen_res["p_value"],
        "quantile_loss": qloss
    }

results_nqr_df = pd.DataFrame(results_nqr).T
print(results_nqr_df)





[NQR-PlanB] step 1500/4638 stored forecasts.
[NQR-PlanB] step 1700/4638 stored forecasts.
[NQR-PlanB] step 1900/4638 stored forecasts.
[NQR-PlanB] step 2100/4638 stored forecasts.
[NQR-PlanB] step 2300/4638 stored forecasts.
[NQR-PlanB] step 2500/4638 stored forecasts.
[NQR-PlanB] step 2700/4638 stored forecasts.
[NQR-PlanB] step 2900/4638 stored forecasts.
[NQR-PlanB] step 3100/4638 stored forecasts.
[NQR-PlanB] step 3300/4638 stored forecasts.
[NQR-PlanB] step 3500/4638 stored forecasts.
[NQR-PlanB] step 3700/4638 stored forecasts.
[NQR-PlanB] step 3900/4638 stored forecasts.
[NQR-PlanB] step 4100/4638 stored forecasts.
[NQR-PlanB] step 4300/4638 stored forecasts.
[NQR-PlanB] step 4500/4638 stored forecasts.
      violation_rate  kupiec_LR  kupiec_p  christoffersen_LR  \
0.01        0.007327   2.496854  0.114074           1.959388   
0.05        0.059255   5.356731  0.020642          17.383119   
0.10        0.115642   8.166738  0.004267           1.437925   

      christoffersen_p 

In [2]:
import torch
print(torch.cuda.is_available())

True
